In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns

from string import punctuation
from nltk.tokenize import word_tokenize
from nltk.stem import LancasterStemmer

from string import punctuation
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem.wordnet import WordNetLemmatizer
import re
import warnings
warnings.filterwarnings('ignore')
import os

In [3]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [4]:
train_data = pd.read_csv('C:/Users/parth/Downloads/SentimentAnaliser_NLP-main/SentimentAnaliser_NLP-main/content/train.csv',encoding='latin1');
test_data = pd.read_csv('C:/Users/parth/Downloads/SentimentAnaliser_NLP-main/SentimentAnaliser_NLP-main/content/test.csv',encoding='latin1');
df = pd.concat([train_data,test_data])

In [5]:
df.head(20)

,textID,text,selected_text,sentiment,Time of Tweet,Age of User,Country,Population -2020,Land Area (Km²),Density (P/Km²)
0,cb774db0d1,"I`d have responded, if I were going","I`d have responded, if I were going",neutral,morning,0-20,Afghanistan,38928346.0,652860.0,60.0
1,549e992a42,Sooo SAD I will miss you here in San Diego!!!,Sooo SAD,negative,noon,21-30,Albania,2877797.0,27400.0,105.0
2,088c60f138,my boss is bullying me...,bullying me,negative,night,31-45,Algeria,43851044.0,2381740.0,18.0
3,9642c003ef,what interview! leave me alone,leave me alone,negative,morning,46-60,Andorra,77265.0,470.0,164.0
4,358bd9e861,"Sons of ****, why couldn`t they put them on t...","Sons of ****,",negative,noon,60-70,Angola,32866272.0,1246700.0,26.0
5,28b57f3990,http://www.dothebouncy.com/smf - some shameles...,http://www.dothebouncy.com/smf - some shameles...,neutral,night,70-100,Antigua and Barbuda,97929.0,440.0,223.0
6,6e0c6d75b1,2am feedings for the baby are fun when he is a...,fun,positive,morning,0-20,Argentina,45195774.0,2736690.0,17.0
7,50e14c0bb8,Soooo high,Soooo high,neutral,noon,21-30,Armenia,2963243.0,28470.0,104.0
8,e050245fbd,Both of you,Both of you,neutral,night,31-45,Australia,25499884.0,7682300.0,3.0
9,fc2cbefa9d,Journey!? Wow... u just became cooler. hehe....,Wow... u just became cooler.,positive,morning,46-60,Austria,9006398.0,82400.0,109.0


In [6]:
def remove_unnecessary_characters(text):
    text = re.sub(r'<.*?>', '', str(text))
    text = re.sub(r'[^a-zA-Z0-9\s]', '', str(text))
    text = re.sub(r'\s+', ' ', str(text)).strip()
    return text
df['clean_text'] = df['text'].apply(remove_unnecessary_characters)

In [7]:
import nltk
from nltk.tokenize import word_tokenize

def tokenize_text(text):
    try:
        text = str(text)
        return word_tokenize(text)
    except Exception as e:
        print(f"Error tokenizing text: {e}")
        return []

# Apply tokenization
df['tokens'] = df['text'].astype(str).apply(tokenize_text)

In [8]:
def normalize_text(text):
    if isinstance(text, str):
        text = text.lower()
        text = re.sub(r'[^\w\s]', '', text)
        text = re.sub(r'\s+', ' ', text).strip()
    else:
        text = str(text)
    return text
df['normalized_text'] = df['text'].apply(normalize_text)

In [9]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))  # load once

def remove_stopwords(text):
    if isinstance(text, str):
        words = text.split()
        filtered_words = [word for word in words if word.lower() not in stop_words]
        return ' '.join(filtered_words)
    return ''

df['text_without_stopwords'] = df['text'].apply(remove_stopwords)

In [10]:
df.dropna(inplace=True)

In [11]:
df['sentiment_code'] = df['sentiment'].astype('category').cat.codes
sentiment_distribution = df['sentiment_code'].value_counts(normalize=True)

In [12]:
stuff_to_be_removed = list(stopwords.words('english'))+list(punctuation)
stemmer = LancasterStemmer()
corpus = df['text'].tolist()
print(len(corpus))
print(corpus[0])

27480
 I`d have responded, if I were going


In [13]:
final_corpus = df['text'].astype(str).tolist()
data_eda = pd.DataFrame()
data_eda['text'] = final_corpus
data_eda['sentiment'] = df["sentiment"].values
data_eda.head()

,text,sentiment
0,"I`d have responded, if I were going",neutral
1,Sooo SAD I will miss you here in San Diego!!!,negative
2,my boss is bullying me...,negative
3,what interview! leave me alone,negative
4,"Sons of ****, why couldn`t they put them on t...",negative


In [14]:
# df['Time of Tweet'] = df['Time of Tweet'].astype('category').cat.codes
# df['Country'] = df['Country'].astype('category').cat.codes
# df['Age of User']=df['Age of User'].replace({'0-20':18,'21-30':25,'31-45':38,'46-60':53,'60-70':65,'70-100':80})

In [15]:
df=df.drop(columns=['textID','Time of Tweet', 'Age of User', 'Country', 'Population -2020', 'Land Area (Km²)', 'Density (P/Km²)'])

In [16]:
import string

def wp(text):
    text = re.sub('https?://\S+|www\.\S+', '', text)
    text = re.sub('<.*?>+', '', text)
    text = re.sub('[%s]' % re.escape(string.punctuation), '', text)
    text = re.sub('\n', '', text)
    text = re.sub('\w*\d\w*', '', text)
    return text

# Assuming df is defined somewhere in your code
df['selected_text'] = df["selected_text"].apply(wp)

In [17]:
X=df['selected_text']
y= df['sentiment']

In [18]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [19]:
from sklearn.model_selection import  GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [20]:
# Text Vectorization with trigram support
vectorizer = TfidfVectorizer(max_features=1500, ngram_range=(1, 3))
X_train_vectorized = vectorizer.fit_transform(X_train)
X_test_vectorized = vectorizer.transform(X_test)

# Define the Random Forest classifier with expanded hyperparameter search
param_grid = {'n_estimators': [200, 400, 600], 'max_depth': [15, 25, 35]}
rf_classifier = RandomForestClassifier(random_state=42)
grid_search = GridSearchCV(rf_classifier, param_grid, cv=3, n_jobs=-1)
grid_search.fit(X_train_vectorized, y_train)

# Make predictions on the test set using the best estimator from grid search
best_rf_classifier = grid_search.best_estimator_
y_pred = best_rf_classifier.predict(X_test_vectorized)

# Evaluate the classifier
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')

print(f'Accuracy: {accuracy:.2f}')
print(f'Precision: {precision:.2f}')
print(f'Recall: {recall:.2f}')
print(f'F1 Score: {f1:.2f}')



Accuracy: 0.72
Precision: 0.74
Recall: 0.72
F1 Score: 0.71


In [21]:
import joblib
joblib.dump(best_rf_classifier, 'sentiment_model.pkl')


['sentiment_model.pkl']

In [22]:
loaded_model = joblib.load('sentiment_model.pkl')

new_text = " I am so angry right now"

new_text_vectorized = vectorizer.transform([new_text])

predicted_sentiment = loaded_model.predict(new_text_vectorized)

print(f'Predicted Sentiment: {predicted_sentiment[0]}')

Predicted Sentiment: negative


In [ ]:
import pickle

# Save model
pickle.dump(best_rf_classifier, open("model.pkl", "wb"))
pickle.dump(vectorizer, open("vectorizer.pkl", "wb"))